# Homodimer Confidence Metric Diagnostic Notebook

**Purpose:** This notebook helps you understand *why* different confidence scores for a predicted protein complex (homodimer) from the AlphaFold Database (AFDB) agree or disagree.

**What is a homodimer?** A homodimer is a protein complex made of two identical copies of the same protein chain (chain A and chain B). AlphaFold can predict the 3D structure of such complexes.

**What are confidence scores?** AlphaFold and related tools compute several numbers that estimate how reliable the predicted structure is. This notebook computes five of them:
- **ipTM** – Overall confidence in chain positioning
- **ipSAE** – Stricter confidence using only well-predicted residues
- **pDockQ** – Structural plausibility (contact count + pLDDT)
- **pDockQ2** – Like pDockQ but also checks PAE at the interface
- **LIS** – Density of low-error inter-chain interactions

**No structural biology background required.** Every section includes a plain-language explanation before any code.

---
**Usage:** Set `ACCESSION_ID` in Section 1, then run all cells (Runtime → Run all).

In [ ]:
# Bootstrap: this one cell works unchanged locally and on Google Colab.
#
# Locally it finds the checkout you are already running from, and touches
# nothing: no clone, no fetch, no reset. On Colab it shallow-clones the public
# repo into /content, refreshing that clone if it is already there. Either way
# it puts `src/` on sys.path and imports the analysis module from there.
#
# Deliberately NOT `pip install`-ing InsightFold itself: that would resolve
# pyproject.toml and drag in biopython, gemmi, scipy and plotly, breaking both
# the project's dependency rules and the 60 s Colab install budget.
# The repo is public, so there is no token, no auth header and no getpass
# (getpass would block forever in a Run-all notebook).
import shutil
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

REPO_URL = 'https://github.com/PDBeurope/InsightFold.git'

# TODO(merge): revert REPO_BRANCH to 'main' (or a release tag) once the
# homodimer-notebook-rework branch merges. `complex_interface_utils` exists only
# on that branch today, so cloning 'main' gives a checkout without it. Grep for
# "TODO(merge)" before releasing this notebook.
REPO_BRANCH = 'homodimer-notebook-rework'

COLAB_CLONE_DIR = Path('/content/InsightFold')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo_root(start=None):
    """Walk up from `start` (default: cwd) for a dir holding both pyproject.toml and src/."""
    s = Path(start) if start is not None else Path.cwd()
    s = s.expanduser().resolve()
    for d in (s, *s.parents):
        if (d / 'pyproject.toml').is_file() and (d / 'src').is_dir():
            return d
    return None


def describe_checkout(root):
    """'branch @ sha' for a git checkout, or a plain note when it is not one."""
    try:
        rev = subprocess.run(['git', '-C', str(root), 'rev-parse', '--abbrev-ref', 'HEAD'],
                             capture_output=True, text=True)
        sha = subprocess.run(['git', '-C', str(root), 'rev-parse', '--short', 'HEAD'],
                             capture_output=True, text=True)
        if rev.returncode == 0 and sha.returncode == 0:
            return f'{rev.stdout.strip()} @ {sha.stdout.strip()}'
    except (OSError, subprocess.SubprocessError):
        pass
    return 'unknown (not a git checkout)'


def ensure_colab_clone(clone_dir, url=REPO_URL, branch=REPO_BRANCH):
    """Clone `url`@`branch` into `clone_dir`, or refresh a clone already there.

    Only ever called with the disposable Colab scratch directory: `git reset
    --hard` must never run against a checkout somebody is working in. A clone
    left over from an earlier session can predate the branch this notebook
    needs, which is why an existing clone is refreshed rather than reused.
    """
    clone_dir = Path(clone_dir)

    if clone_dir.exists() and not (clone_dir / '.git').is_dir():
        # Something is in the way that is not a clone. Move it aside rather than
        # delete it, so nothing of the user's is destroyed silently.
        aside = clone_dir.with_name(clone_dir.name + '.not-a-git-repo')
        shutil.rmtree(aside, ignore_errors=True)
        print(f'{clone_dir} exists but is not a git clone; moving it to {aside}')
        clone_dir.rename(aside)

    if not (clone_dir / '.git').is_dir():
        print(f'Cloning {url} ({branch}) ...')
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', branch, url, str(clone_dir)],
            check=True,
        )
        return

    print(f'Refreshing existing clone at {clone_dir} -> {branch} ...')

    def _git(*args):
        return subprocess.run(['git', '-C', str(clone_dir), *args],
                              capture_output=True, text=True)

    for step in (('fetch', '--depth', '1', 'origin', branch),
                 ('reset', '--hard', 'FETCH_HEAD')):
        done = _git(*step)
        if done.returncode != 0:
            # Offline, or a clone we cannot reason about: an existing checkout is
            # better than a hard failure, but say so loudly.
            detail = (done.stderr or done.stdout).strip().splitlines()
            print(f'  git {step[0]} failed; falling back to the existing checkout as-is.')
            if detail:
                print(f'  git said: {detail[-1]}')
            return
    print(f'  now at {describe_checkout(clone_dir)}')


# Search from the working directory only. The Colab clone is handled separately
# below so that a stale clone gets refreshed instead of being picked up as-is.
REPO_ROOT = find_repo_root()

if IN_COLAB and (REPO_ROOT is None or REPO_ROOT == COLAB_CLONE_DIR.resolve()):
    ensure_colab_clone(COLAB_CLONE_DIR)
    REPO_ROOT = find_repo_root(COLAB_CLONE_DIR)

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the InsightFold checkout.\n'
        f'Looked upwards from {Path.cwd()} for a directory containing both '
        "'pyproject.toml' and 'src/'.\n"
        'Run this notebook from inside a clone of '
        'https://github.com/PDBeurope/InsightFold, for example:\n'
        f'    git clone -b {REPO_BRANCH} https://github.com/PDBeurope/InsightFold.git\n'
        '    cd InsightFold && jupyter lab notebooks/homodimer_diagnostic.ipynb'
    )

SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# molviewspec powers the optional 3D views in Section 6. Install it only when it
# is genuinely missing, and only on Colab: locally it comes from the environment,
# and a pip call on every run is pure latency.
if find_spec('molviewspec') is None and IN_COLAB:
    print('Installing molviewspec ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'molviewspec'],
        check=False,
    )

HAS_MOLVIEWSPEC = find_spec('molviewspec') is not None

REVISION = describe_checkout(REPO_ROOT)

try:
    from insightfold import complex_interface_utils as ciu  # noqa: E402
except ImportError as exc:
    _remedy = (
        f'On Colab, remove the clone and re-run this cell to fetch a fresh copy:\n'
        f'    !rm -rf {COLAB_CLONE_DIR}\n'
        if IN_COLAB else
        f'Locally, update your checkout:\n'
        f'    git -C {REPO_ROOT} switch {REPO_BRANCH} && git -C {REPO_ROOT} pull\n'
    )
    raise ImportError(
        f"Could not import 'insightfold.complex_interface_utils': {exc}\n"
        f'  repo root in use: {REPO_ROOT}\n'
        f'  revision in use:  {REVISION}\n'
        f'  branch expected:  {REPO_BRANCH}\n'
        'The likely cause is a checkout that predates this module or sits on a '
        f"different branch: complex_interface_utils.py lives on '{REPO_BRANCH}'.\n"
        + _remedy
    ) from exc

print(f'Environment: {"Colab" if IN_COLAB else "local"}')
print(f'Repo root:   {REPO_ROOT}')
print(f'Revision:    {REVISION}')
print(f'molviewspec: {"available" if HAS_MOLVIEWSPEC else "not installed (Section 6 will be skipped)"}')

In [ ]:
%matplotlib inline
import json

import numpy as np
from IPython.display import HTML, display

# Every formula, threshold, figure and 3D view now lives in the module the
# bootstrap cell imported as `ciu`; this notebook keeps only the narrative and
# the orchestration. Styling is an explicit call because importing a module must
# never restyle somebody else's figures as a side effect.
ciu.apply_plot_style()

print('Imports OK')


---
## Section 1 — Setup and Data Loading

**What happens here?**
We fetch all the raw data from the AlphaFold Database (AFDB) for one homodimer accession.

The AFDB stores:
- The **3D structure** in mmCIF format (atom positions for every residue)
- The **PAE matrix** — a grid of numbers where entry `[i, j]` = how uncertain AlphaFold is about the position of residue `i` given that residue `j` is aligned correctly (lower = better)
- The **pLDDT scores** — a per-residue confidence score (0–100, higher = better)

We parse all three into data structures we can analyse.

In [ ]:

# ── USER INPUT ────────────────────────────────────────────────────────
ACCESSION_ID = 'AF-0000000065889468'  # @param {type:"string"}
# ───────────────────────────────────────────────────────────────────

# Set True to upload local files instead of fetching from AFDB
USE_LOCAL_FILE = False

# The cutoffs are the module's, so the scores, the figures and the printed
# summaries can never disagree about which value is in force.
PAE_CUTOFF  = ciu.PAE_CUTOFF    # 10.0 Å — AFDB standard cutoff for ipSAE
DIST_CUTOFF = ciu.DIST_CUTOFF   #  8.0 Å — CB-CB contact cutoff
LIS_CUTOFF  = ciu.LIS_CUTOFF    # 12.0 Å — PAE cutoff for LIS


In [ ]:
import ipywidgets as _widgets
from IPython.display import display as _display

if USE_LOCAL_FILE:
    _upload_cif   = _widgets.FileUpload(accept='.cif,.mmcif', multiple=False,
                                        description='mmCIF (required)')
    _upload_pae   = _widgets.FileUpload(accept='.json',       multiple=False,
                                        description='PAE JSON (optional)')
    _upload_plddt = _widgets.FileUpload(accept='.json',       multiple=False,
                                        description='pLDDT JSON (optional)')
    _display(_widgets.VBox([
        _widgets.HTML('<b>Upload local structure files, then run the next cell.</b>'),
        _upload_cif,
        _upload_pae,
        _upload_plddt,
    ]))
else:
    print('Online mode — files will be downloaded from AFDB.')

In [ ]:
if USE_LOCAL_FILE:
    prediction = None
    print(f'Local file mode — skipping AFDB API for {ACCESSION_ID}')
else:
    print(f'Fetching: {ciu.AFDB_PREDICTION_URL.format(accession=ACCESSION_ID)}')
    prediction = ciu.fetch_afdb_metadata(ACCESSION_ID)
    # The endpoint returns one entry per chain, in a non-deterministic order.
    # `AFDBPrediction` keeps all of them; `primary_entry` is the whole-complex
    # display entry that R020 replaces with a per-chain lookup.
    print(f'Chains described: {list(prediction.chain_ids)}')
    print('Available fields:', sorted(prediction.primary_entry.keys()))


In [ ]:
def _read_upload(widget):
    """Return bytes from a FileUpload widget, or None if nothing was uploaded."""
    if not widget.value:
        return None
    return bytes(widget.value[0]['content'])


if USE_LOCAL_FILE:
    _cif_bytes = _read_upload(_upload_cif)
    if _cif_bytes is None:
        raise ValueError('No mmCIF file uploaded — use the upload widget in the cell above.')
    cif_text = _cif_bytes.decode('utf-8', errors='replace')
    print(f'Loaded mmCIF  : {_upload_cif.value[0]["name"]}')

    _pae_bytes = _read_upload(_upload_pae)
    pae_raw = json.loads(_pae_bytes) if _pae_bytes else None
    print(f'Loaded PAE    : {_upload_pae.value[0]["name"]}' if _pae_bytes else
          'PAE file not uploaded — PAE-dependent analyses will be skipped.')

    _plddt_bytes = _read_upload(_upload_plddt)
    plddt_raw = json.loads(_plddt_bytes) if _plddt_bytes else None
    print(f'Loaded pLDDT  : {_upload_plddt.value[0]["name"]}' if _plddt_bytes else
          'pLDDT file not uploaded — pLDDT analyses will be skipped.')
else:
    print(f'Downloading mmCIF: {prediction.cif_url}')
    cif_text  = ciu.download_structure(prediction)
    print(f'Downloading PAE:   {prediction.pae_url}')
    pae_raw   = ciu.download_pae(prediction)
    print(f'Downloading pLDDT: {prediction.plddt_url}')
    plddt_raw = ciu.download_plddt(prediction)

print('All downloads complete.')


In [ ]:
# mmCIF parsing, CB/CA selection and the GLY fallback all live in the module.
chains = ciu.parse_structure(cif_text)
chain_ids = sorted(chains)

print(f'Chains found: {chain_ids}')
for _chain_id in chain_ids:
    print(f'  Chain {_chain_id}: {chains[_chain_id].n_residues} residues')


In [ ]:
_structure_lengths = {cid: chain.n_residues for cid, chain in chains.items()}
pae   = ciu.parse_pae(pae_raw, fallback_lengths=_structure_lengths)
plddt = ciu.parse_plddt(plddt_raw, fallback_lengths=_structure_lengths)

# A chain-length disagreement would misalign every quadrant slice and produce
# plausible but wrong scores, so it is checked rather than assumed.
ciu.verify_chain_lengths(chains, pae)
ciu.verify_chain_lengths(chains, plddt)

# D4: everything downstream takes one explicit ordered chain pair. Nothing here
# hard-codes 'A' and 'B'; PAE is asymmetric, so the order is part of the query.
chain_x, chain_y = pae.chain_ids[0], pae.chain_ids[1]
pair    = pae.ordered_pair(chain_x, chain_y)
plddt_x = plddt.for_chain(chain_x)
plddt_y = plddt.for_chain(chain_y)
nx, ny  = pair.nx, pair.ny

print(f'PAE matrix shape: {pae.matrix.shape}')
for _span in pae.spans:
    print(f'  chain {_span.chain_id}: residues {_span.start}–{_span.end} '
          f'({_span.length} total) {_span.name}')
print(f'PAE quadrants: {chain_x}{chain_x}={pair.block_xx.shape}, '
      f'{chain_x}{chain_y}={pair.block_xy.shape}, '
      f'{chain_y}{chain_x}={pair.block_yx.shape}, '
      f'{chain_y}{chain_y}={pair.block_yy.shape}')
print(f'pLDDT: chain {chain_x} mean={plddt_x.mean():.1f}, '
      f'chain {chain_y} mean={plddt_y.mean():.1f}')


In [ ]:
# TODO(R020): `primary_entry` is the endpoint's first entry, which for a
# heterodimer describes one chain and is reported here as if it described the
# complex. R020 replaces it with a per-chain lookup.
_entry = prediction.primary_entry if prediction is not None else {}

print('=' * 55)
print('HOMODIMER DIAGNOSTIC REPORT')
print('=' * 55)
print(f'Accession ID  : {ACCESSION_ID}')
print(f'UniProt ID    : {_entry.get("uniprotAccession", "N/A")}')
print(f'Organism      : {_entry.get("organismScientificName", _entry.get("organism", "N/A"))}')
print(f'Protein name  : {_entry.get("proteinFullName", _entry.get("uniprotDescription", "N/A"))}')
print(f'Gene name     : {_entry.get("geneNames", "N/A")}')
print(f'Monomer length: {len(_entry.get("sequence", ""))} residues')
print(f'Dimer length  : {nx + ny} residues total '
      f'(chain {chain_x}: {nx}, chain {chain_y}: {ny})')
print(f'Model version : {_entry.get("modelVersion", "N/A")}')
print('=' * 55)


---
## Section 2 — Interface Detection

**What is the interface?**
The interface is the region where the two protein chains touch each other. We identify it by measuring distances between atoms: if a residue from chain A has a beta-carbon (CB) within **8.0 Å** of a CB atom in chain B, those two residues are in "contact" and are part of the interface.

*(We use the beta-carbon CB because it better represents the side chain position. For glycine, which has no CB, we use the alpha-carbon CA instead.)*

**Why does this matter?**
The interface residues are the ones where the two chains actually interact. Many confidence metrics focus specifically on these residues — if AlphaFold is uncertain about the interface, that's a red flag.

In [ ]:
contacts = ciu.detect_interface(chains[chain_x], chains[chain_y],
                                dist_cutoff=DIST_CUTOFF)

print(f'Contact cutoff         : {contacts.dist_cutoff} Å (CB-CB; CA for GLY)')
print(f'Number of contact pairs: {contacts.n_contact_pairs}')
print(f'Interface residues {chain_x}   : {contacts.n_interface_residues_x} / {nx} '
      f'({100 * contacts.n_interface_residues_x / nx:.1f}%)')
print(f'Interface residues {chain_y}   : {contacts.n_interface_residues_y} / {ny} '
      f'({100 * contacts.n_interface_residues_y / ny:.1f}%)')

_if_res_x = chains[chain_x].res_ids[contacts.mask_x]
if _if_res_x.size:
    print(f'Interface residues {chain_x}   : '
          f'{_if_res_x[0]} – {_if_res_x[-1]} (range)')


In [ ]:
display(ciu.plot_interface_contact_map(contacts))


---
## Section 3 — PAE Matrix Decomposition

**What is the PAE matrix?**
The Predicted Aligned Error (PAE) matrix is an N×N grid where N = total number of residues in both chains combined. Entry `[i, j]` = AlphaFold's estimate of how wrong the position of residue `i` would be, *if* we fixed residue `j` and rotated/translated everything else to align it.

- **Low PAE (blue)** → AlphaFold is confident about the relative position
- **High PAE (red)** → AlphaFold is uncertain

For a homodimer, the matrix has four quadrants:
- **Top-left (AA)**: confidence within chain A
- **Bottom-right (BB)**: confidence within chain B
- **Top-right (AB)** and **bottom-left (BA)**: confidence *between* chains — this is what all inter-chain scores use

Each score uses a different subset of the inter-chain region. The panel below shows which cells each score uses.

In [ ]:
display(ciu.plot_pae_matrix(pae, chain_x, chain_y, accession=ACCESSION_ID))


In [ ]:
display(ciu.plot_pae_score_masks(pair, contacts, max_pae=pae.max_pae,
                                 pae_cutoff=PAE_CUTOFF, lis_cutoff=LIS_CUTOFF))

# Same masks the panels are drawn from, so the figure and the numbers agree.
_masks = ciu.score_masks(pair, contacts,
                         pae_cutoff=PAE_CUTOFF, lis_cutoff=LIS_CUTOFF)
_block_cells = pair.block_xy.size
print(f'Cells of the {chain_x}→{chain_y} block used by each score:')
for _name, _mask in _masks.items():
    print(f'  {_name:12s}: {int(_mask.sum()):6d} cells '
          f'({100 * int(_mask.sum()) / _block_cells:.1f}%)')


---
## Section 4 — Score Computation and Decomposition

**What we do here:** Compute all five confidence scores from scratch, showing every intermediate value.

All PAE-based scores use a **TM-score function**: `ptm(x, d0) = 1 / (1 + (x/d0)²)`. This converts a PAE value `x` into a score from 0 (bad) to 1 (perfect). The parameter `d0` controls how steeply the score falls off with increasing PAE.

The `d0` is computed from `L` (number of relevant residues): `d0(L) = max(1.0, 1.24 × (L−15)^(1/3) − 1.8)`.

Scores that use *more* residues for normalisation have a *larger* `d0`, making them more lenient. This is why the three ipSAE variants (d0res, d0chn, d0dom) can give very different values.

In [ ]:
res_iptm    = ciu.compute_iptm_d0chn(pair)
res_ipsae   = ciu.compute_ipsae(pair, pae_cutoff=PAE_CUTOFF)
res_pdockq  = ciu.compute_pdockq(contacts, plddt_x, plddt_y)
res_pdockq2 = ciu.compute_pdockq2(contacts, pair, plddt_x, plddt_y)
res_lis     = ciu.compute_lis(pair, lis_cutoff=LIS_CUTOFF)

# Keyed by `ciu.THRESHOLDS` key, so the summary table, the agreement matrix and
# every traffic light below read one set of names.
scores = {
    'ipsae_d0res': res_ipsae.d0res.score,
    'ipsae_d0chn': res_ipsae.d0chn.score,
    'ipsae_d0dom': res_ipsae.d0dom.score,
    'iptm_d0chn':  res_iptm.score,
    'pdockq':      res_pdockq.score,
    'pdockq2':     res_pdockq2.score,
    'lis':         res_lis.score,
}

print('\n── Score Results ──────────────────────────────')
for _name, _value in scores.items():
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_name]:18s}: {_value:.4f}')

# Every score is reported for one direction of the pair; the module keeps both,
# so the intermediates below name the direction that supplied the number.
_pdockq2_dir = res_pdockq2.winning_direction

print('\n── Intermediate Values ───────────────────────')
print(f'  pDockQ   n_contacts : {res_pdockq.n_contact_pairs}')
print(f'  pDockQ   mean_pLDDT : {res_pdockq.mean_plddt:.2f}')
print(f'  pDockQ   x          : {res_pdockq.x:.4f}')
print(f'  pDockQ2  mean_ptm   : {_pdockq2_dir.mean_ptm:.4f}')
print(f'  pDockQ2  mean_pLDDT : {_pdockq2_dir.mean_plddt:.2f}')
print(f'  pDockQ2  x          : {_pdockq2_dir.x:.4f}')
print(f'  LIS      n_valid_{chain_x}{chain_y} : {res_lis.forward.n_valid_pairs}')
print(f'  LIS      n_valid_{chain_y}{chain_x} : {res_lis.reverse.n_valid_pairs}')
print(f'  ipTM     d0_chn     : {res_iptm.d0:.4f}')
print(f'  ipTM     best_res_{chain_x}{chain_y}: {res_iptm.forward.argmax_index}')
print(f'  ipSAE    d0_chn     : {res_ipsae.d0chn_value:.4f}')
print(f'  ipSAE    d0_dom     : {res_ipsae.d0dom.d0:.4f}  '
      f'(n_dom={res_ipsae.d0dom.n0})')


In [ ]:
display(ciu.plot_residue_score_profiles(res_iptm, res_ipsae, contacts,
                                        plddt_x, plddt_y))


---
## Section 5 — Per-Residue pLDDT at the Interface

**What is pLDDT?**
pLDDT (predicted Local Distance Difference Test) is AlphaFold's confidence score for each individual residue, ranging from 0 to 100. A score above 90 means AlphaFold is very confident about that residue's position within its local neighbourhood; below 50 means the residue is likely disordered.

**Why look at interface pLDDT separately?**
pDockQ scores depend on the pLDDT of interface residues. If the interface contains many low-pLDDT residues, pDockQ will be dragged down even if the structural contacts look physically reasonable. Comparing interface vs. non-interface pLDDT distributions helps us understand whether a low pDockQ reflects a *globally* uncertain protein or specifically uncertain *interface* residues.

**AlphaFold colour scheme:** dark blue (>90) → light blue (70–90) → yellow (50–70) → orange (<50).

In [ ]:
if_plddt = np.concatenate([plddt_x[contacts.mask_x], plddt_y[contacts.mask_y]])
ni_plddt = np.concatenate([plddt_x[~contacts.mask_x], plddt_y[~contacts.mask_y]])
n_low_if = int((if_plddt < 70).sum())

print(f'Interface pLDDT   mean={if_plddt.mean():.1f}, median={np.median(if_plddt):.1f}')
print(f'Non-interface     mean={ni_plddt.mean():.1f}, median={np.median(ni_plddt):.1f}')
print(f'Low-pLDDT (<70) interface residues: {n_low_if} / {len(if_plddt)} '
      f'({100 * n_low_if / max(len(if_plddt), 1):.1f}%)')

display(ciu.plot_plddt_distribution(contacts, plddt_x, plddt_y))


---
## Section 6 — 3D Structure Visualisation (MolViewSpec)

**What is MolViewSpec?**
MolViewSpec is a format for describing 3D molecular visualisations. We use it to colour the protein structure by different diagnostic criteria and then view it in **Mol***, the same interactive 3D viewer used by the PDBe and RCSB databases.

We generate four views:
1. **Overview** — chains coloured teal (A) and coral (B), interface in amber
2. **pLDDT mapping** — residues coloured by confidence (AlphaFold colour scheme)
3. **Interface diagnostic** — interface residues coloured by per-residue ipSAE score
4. **Disagreement** — residues where PAE-based (ipSAE) and contact-based (pDockQ) signals diverge

Each view generates a link to open in the Mol* web viewer.

In [ ]:
if not ciu.molviewspec_available():
    print(ciu.MOLVIEWSPEC_MISSING_MESSAGE)
elif prediction is None:
    print('Skipping 3D views — Mol* downloads the structure itself, so local '
          'file mode has no URL to hand it.')
else:
    _source = ciu.resolve_structure_source(prediction)

    # Built lazily so that one failing view does not cost you the other three.
    _views = [
        ('chain_overview',
         lambda: ciu.build_chain_overview_view(_source, chains, contacts)),
        ('plddt',
         lambda: ciu.build_plddt_view(_source, chains, contacts,
                                      plddt={chain_x: plddt_x, chain_y: plddt_y})),
        ('interface_value',
         lambda: ciu.build_interface_value_view(_source, chains, contacts,
                                                res_ipsae.d0res.forward.values,
                                                res_ipsae.d0res.reverse.values)),
        ('disagreement',
         lambda: ciu.build_disagreement_view(_source, chains, contacts,
                                             res_ipsae.d0res.forward.values)),
    ]

    for _key, _build in _views:
        _label = ciu.MVS_VIEW_LABELS[_key]
        try:
            ciu.show_mol_view(_build(), _label)
        except Exception as exc:
            print(f'{_label} failed: {exc}')


---
## Section 7 — Diagnostic Summary

**What we do here:** Bring together all five scores into a single interpretable diagnostic.

**Traffic-light thresholds** (based on AFDB filtering criteria and literature):
- **Green (high confidence):** ipSAE_d0res ≥ 0.6, ipTM ≥ 0.6, pDockQ ≥ 0.23, pDockQ2 ≥ 0.15, LIS ≥ 0.3
- **Amber (moderate):** half of the above thresholds
- **Red (low confidence):** below amber

These thresholds are starting heuristics; see the references for more detail.

In [ ]:
descriptions = {
    'ipsae_d0res': 'Primary AFDB classifier. Inter-chain PAE, d0 from valid-pair count.',
    'ipsae_d0chn': 'Conservative variant. Penalises small interfaces more.',
    'ipsae_d0dom': 'Intermediate variant. d0 from domain-level residue count.',
    'iptm_d0chn':  'Global inter-chain confidence. All PAE cells used, no cutoff.',
    'pdockq':      'Structural plausibility. Contact count x interface pLDDT. No PAE.',
    'pdockq2':     'Contact PAE + interface pLDDT. Bridges pDockQ and ipSAE.',
    'lis':         'Density of inter-chain PAE < 12. Quantity of interaction.',
}
colour_hex = {'green': '#4CAF50', 'amber': '#FF9800', 'red': '#F44336'}

# Bands and colours come from `ciu.THRESHOLDS` via `ciu.traffic_light`, the one
# canonical table. ipSAE_d0res carries AlphaFold DB's four published band names;
# the other six use the three-colour scheme.
rows = ''
for _name, _value in scores.items():
    _colour, _band = ciu.traffic_light(_value, _name)
    rows += (
        f'<tr>'
        f'<td style="font-weight:bold;padding:6px 12px;">'
        f'{ciu.SCORE_DISPLAY_NAMES[_name]}</td>'
        f'<td style="padding:6px 12px;text-align:center;font-size:1.15em;font-weight:bold;">'
        f'{_value:.4f}</td>'
        f'<td style="padding:6px 12px;text-align:center;">'
        f'<span style="background:{colour_hex[_colour]};color:white;padding:3px 10px;'
        f'border-radius:12px;font-weight:bold;font-size:0.85em;">{_band}</span></td>'
        f'<td style="padding:6px 12px;font-size:0.88em;color:#555;">'
        f'{descriptions[_name]}</td>'
        f'</tr>'
    )

display(HTML(
    f'<h3>Confidence Score Summary — {ACCESSION_ID}</h3>'
    f'<table style="border-collapse:collapse;width:100%;font-family:sans-serif;">'
    f'<thead><tr style="background:#f5f5f5;">'
    f'<th style="padding:8px 12px;text-align:left;">Score</th>'
    f'<th style="padding:8px 12px;">Value</th>'
    f'<th style="padding:8px 12px;">Confidence</th>'
    f'<th style="padding:8px 12px;text-align:left;">What it measures</th>'
    f'</tr></thead><tbody>{rows}</tbody></table>'
))


In [ ]:
display(ciu.plot_score_agreement(scores))


In [ ]:
import textwrap

# The five independent values; the other two ipSAE variants are the same
# measurement under a more forgiving normalisation, not extra evidence.
score_names = list(ciu.AGREEMENT_SCORES)
lights = {name: ciu.traffic_light(scores[name], name) for name in score_names}

ipsae_val   = scores['ipsae_d0res']
pdockq_val  = scores['pdockq']
pdockq2_val = scores['pdockq2']
lis_val     = scores['lis']

tl_ipsae  = lights['ipsae_d0res'][0]
tl_pdockq = lights['pdockq'][0]
tl_lis    = lights['lis'][0]

n_green = sum(1 for name in score_names if lights[name][0] == 'green')
n_red   = sum(1 for name in score_names if lights[name][0] == 'red')

statements = []

if n_green >= 4:
    statements.append(
        'OVERALL: This complex has consistently HIGH confidence across all metrics. '
        'The interface is well-resolved (high pLDDT), structurally plausible (high pDockQ), '
        'and the PAE matrix shows strong inter-chain confidence.')
elif n_red >= 4:
    statements.append(
        'OVERALL: This complex has LOW confidence across most metrics. '
        'The predicted interaction may be unreliable. Treat structural conclusions with caution.')
else:
    statements.append(
        'OVERALL: Mixed confidence signals — scores disagree. See details below.')

# ipSAE high, pDockQ low
if tl_ipsae == 'green' and tl_pdockq in ('amber', 'red'):
    statements.append(
        'PAE vs STRUCTURE: AlphaFold is confident about relative chain positioning (PAE; '
        f'ipSAE={ipsae_val:.3f}), but there are few physical contacts at the interface '
        f'(pDockQ={pdockq_val:.3f}). This can occur when chains interact via a small, '
        'tight interface or when the predicted inter-chain distance is slightly too large '
        'for contacts to form under the 8 A cutoff.')

# pDockQ high, ipSAE low
if tl_pdockq == 'green' and tl_ipsae in ('amber', 'red'):
    statements.append(
        'STRUCTURE vs PAE: The interface has many contacts between well-resolved residues '
        f'(pDockQ={pdockq_val:.3f}), but AlphaFold PAE indicates uncertainty about the '
        f'relative chain arrangement (ipSAE={ipsae_val:.3f}). The local structure of each '
        'chain may be well-predicted even though the docking orientation is uncertain.')

# pDockQ2 vs pDockQ, each in units of its own canonical green threshold
pdockq_norm  = pdockq_val / ciu.THRESHOLDS['pdockq'].green
pdockq2_norm = pdockq2_val / ciu.THRESHOLDS['pdockq2'].green
if abs(pdockq_norm - pdockq2_norm) > 0.4:
    direction = 'pDockQ2 < pDockQ' if pdockq2_norm < pdockq_norm else 'pDockQ2 > pDockQ'
    statements.append(
        f'pDockQ vs pDockQ2 ({direction}): These two scores diverge, indicating that '
        'although physical contacts exist, the PAE confidence at those specific contact '
        'points is '
        + ('low' if pdockq2_norm < pdockq_norm else 'high') +
        '. pDockQ2 incorporates PAE at the interface and is the more informative of the two.')

# LIS high but ipSAE low
if tl_lis == 'green' and tl_ipsae in ('amber', 'red'):
    statements.append(
        f'LIS vs ipSAE: Many inter-chain PAE values are below {LIS_CUTOFF:.0f} A (LIS='
        f'{lis_val:.3f}), but when the cutoff is tightened to {PAE_CUTOFF:.0f} A and the '
        f'TM-score formula is applied (ipSAE={ipsae_val:.3f}), confidence drops. '
        'This suggests a broad but diffuse interaction rather than a tight, '
        'well-defined interface.')

print('=' * 65)
print('DIAGNOSTIC INTERPRETATION')
print('=' * 65)
for _statement in statements:
    print()
    print(textwrap.fill(_statement, width=63))
print()
print('─' * 65)
print('Score summary:')
for _name in score_names:
    print(f'  {ciu.SCORE_DISPLAY_NAMES[_name]:18s}: {scores[_name]:.4f}  '
          f'[{lights[_name][1]}]')
print('─' * 65)
print('Interface statistics:')
print(f'  Contact pairs     : {contacts.n_contact_pairs}')
print(f'  Interface res ({chain_x}) : {contacts.n_interface_residues_x}')
print(f'  Interface res ({chain_y}) : {contacts.n_interface_residues_y}')
print(f'  Mean pLDDT (if)   : {if_plddt.mean():.1f}')
print(f'  Low pLDDT (<70) if: {n_low_if} / {len(if_plddt)}')
print('=' * 65)
print()
print('References:')
print('  ipSAE : Dunbrack Lab (2025) biorxiv 2025.02.10.637595')
print('  pDockQ: Bryant et al. (2022) Nat Commun s41467-022-28865-w')
print('  pDockQ2: Zhu et al. (2023) Bioinformatics btad424')
print('  LIS   : Kim et al. (2024) biorxiv 2024.02.19.580970')
print('  AFDB bands / joint criterion: Han, Tsenkov, Venanzi et al. (2026) '
      'biorxiv 10.64898/2026.03.27.714458v2')
